[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [1]:
# Install the latest torch-judge from this repo in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/CharlesShang/TorchCode.git@master')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.9 MB/s eta 0:00:00


In [11]:
import torch
import torch.nn as nn
import math
# help(torch.einsum)

In [8]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = self.d_model // self.num_heads
        assert self.d_k * self.num_heads == self.d_model, f"{self.d_model} {self.num_heads}"
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x_q, x_kv):
        # pass  # Q from x_q, K/V from x_kv, no causal mask
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
        B, S_q, D = x_q.shape
        B, S_kv, D = x_kv.shape
        q = self.W_q(x_q).reshape(B, S_q, -1, self.d_k).permute(0, 2, 1, 3)
        k = self.W_k(x_kv).reshape(B, S_kv, -1, self.d_k).permute(0, 2, 1, 3)
        v = self.W_v(x_kv).reshape(B, S_kv, -1, self.d_k).permute(0, 2, 1, 3)
        attn = (torch.einsum("bhqd,bhkd->bhqk", q, k)) / math.sqrt(self.d_k)
        attn = torch.softmax(attn, dim=-1)
        y = torch.einsum("bhqk,bhkd->bhqd", attn, v)
        y = y.permute(0, 2, 1, 3).reshape(B, S_q, D)
        return self.W_o(y)

In [9]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

Output: torch.Size([2, 6, 64])


In [10]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (2.2ms)
  ✅ [2/4] Q and KV different lengths (1.1ms)
  ✅ [3/4] No causal mask — all KV affects all Q (46.1ms)
  ✅ [4/4] Gradient flow (82.6ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (132.1ms total)
  Progress saved. Run status() to see your dashboard.

